# Threads Scraper Analysis Stack 

---

## 架構總覽

```
ScrapeCreators API (/v1/threads/*)
        │  HTTP GET + x-api-key
        ▼
┌─ threads_client.py ──────────────────────┐
│  ThreadsClient._request()   (重試 + 錯誤處理)  │
│  ├─ search_posts()    → list[ThreadsPost]     │
│  ├─ get_profile()     → ThreadsProfile        │
│  ├─ get_user_posts()  → list[ThreadsPost]     │
│  ├─ get_post()        → ThreadsPost           │
│  └─ search_users()    → list[dict]            │
│                                               │
│  ThreadsPost.from_api_response()              │
│  └─ 原始 JSON → 正規化 dataclass               │
└───────────────┬───────────────────────┘
                ▼
┌─ keyword_monitor.py ─────────────────────┐
│  KeywordMonitor                               │
│  ├─ search_keyword()  (去重 by post_id)       │
│  ├─ run_round()       (批次搜多關鍵字)         │
│  └─ export_results()  (CSV + JSON → data/)    │
└───────────────┬───────────────────────┘
                ▼
┌─ Notebook (本檔案) ──────────────────────┐
│  載入 → 語言過濾 → 受眾分群 → 品牌分析        │
└──────────────────────────────────────────┘
```

## Pipeline 節點

| # | 節點 | 說明 |
|---|------|------|
| 1 | 環境設置 | 匯入套件、設定 |
| 2 | API 串接 | 初始化 ThreadsClient、驗證連線 |
| 3 | 爬蟲執行 | 用 KeywordMonitor 批次爬取關鍵字貼文 |
| 4 | 資料 Extract | 檢視 API 原始回傳 → dataclass 解析 → 匯出 |
| 5 | 載入 & 過濾 | 讀取 CSV、繁體中文過濾、清洗 |
| 6 | 受眾輪廓 | 高相關貼文、活躍帳號、關鍵字互動 |
| 7 | 受眾分群 | 求推薦 vs 分享心得、品牌/裝備品類分析 |
| 8 | 洞察總結 | 受眾報告 |

## 節點 1：環境設置

匯入所有需要的套件，包括本專案的 `threads_client` 和 `keyword_monitor`。

In [1]:
import os
import sys
from pathlib import Path
DATA_DIR = Path("data")
# 將 files/ 加入模組搜尋路徑
sys.path.insert(0, str(Path("files").resolve()))
print(f"  - 輸出目錄: {DATA_DIR.resolve()}")

  - 輸出目錄: I:\Fnte Workdir\threads-scraper-stack\jp_car_service\data


In [2]:

import json
import re
import time

from collections import Counter
from datetime import datetime

import pandas as pd
import numpy as np

# 匯入自定模組
from threads_client import (
    ThreadsClient, ThreadsPost, ThreadsProfile,
    posts_to_dicts, save_json, save_csv,
)
from keyword_monitor import KeywordMonitor


In [3]:
from dotenv import load_dotenv
load_dotenv(Path(".env"), override=True)

True

## 節點 2：API 串接

初始化 `ThreadsClient`，驗證 API Key 與連線狀態。

**API 架構：**
- 底層使用 `requests.Session` 保持連線
- 所有請求透過 `_request()` 統一處理：自動重試（429/500）、錯誤分類（401/402/400）
- Header 帶 `x-api-key` 認證

In [4]:
# ── 初始化 API Client ──
# API Key 設定
API_KEY = os.environ.get("SCRAPECREATORS_API_KEY")

client = ThreadsClient(api_key=API_KEY, timeout=30, max_retries=3)
print("✔ ThreadsClient 初始化成功")
print(f"  - Base URL: https://api.scrapecreators.com")
print(f"  - Timeout: 30s, Max retries: 3")

# 驗證連線：查詢剩餘額度
try:
    balance = client.get_credit_balance()
    print(f"  - API 額度剩餘: {balance} credits")
except Exception as e:
    print(f"  - ⚠ 額度查詢失敗: {e}")

✔ ThreadsClient 初始化成功
  - Base URL: https://api.scrapecreators.com
  - Timeout: 30s, Max retries: 3
  - API 額度剩餘: 24114 credits


In [5]:
# # ── 逐一測試 5 個 API Endpoint ──
# # 每次呼叫消耗 1 credit

# print("=" * 55)
# print("  測試 5 個 Threads API Endpoints")
# print("=" * 55)

# # ── Endpoint 1: 搜尋貼文（核心爬蟲 endpoint）──
# print("\n── 1. search_posts('AI') ──")
# print("   GET /v1/threads/search?query=AI")
# test_posts = client.search_posts(query="AI")
# print(f"   回傳 {len(test_posts)} 筆貼文")
# if test_posts:
#     p = test_posts[0]
#     print(f"   範例: @{p.username} | {p.like_count} likes")
#     print(f"   Text: {p.text[:80]}...")

# # ── Endpoint 2: 用戶 Profile ──
# print("\n── 2. get_profile('zuck') ──")
# print("   GET /v1/threads/profile?handle=zuck")
# try:
#     profile = client.get_profile("zuck")
#     print(f"   @{profile.username} | {profile.full_name}")
#     print(f"   Followers: {profile.follower_count:,} | Verified: {profile.is_verified}")
# except Exception as e:
#     print(f"   Error: {e}")

# # ── Endpoint 3: 用戶貼文列表 ──
# print("\n── 3. get_user_posts('zuck') ──")
# print("   GET /v1/threads/user/posts?handle=zuck")
# try:
#     user_posts = client.get_user_posts("zuck")
#     print(f"   回傳 {len(user_posts)} 筆貼文")
# except Exception as e:
#     print(f"   Error: {e}")

# # ── Endpoint 4: 搜尋用戶 ──
# print("\n── 4. search_users('tech') ──")
# print("   GET /v1/threads/search/users?query=tech")
# try:
#     users = client.search_users("tech")
#     print(f"   回傳 {len(users)} 位用戶")
#     for u in users[:3]:
#         print(f"   @{u.get('username', '?')} — {u.get('follower_count', '?')} followers")
# except Exception as e:
#     print(f"   Error: {e}")

# # ── Endpoint 5: 單篇貼文詳情 ──
# print("\n── 5. get_post(url) ──")
# print("   GET /v1/threads/post?url=...")
# if test_posts and test_posts[0].permalink:
#     try:
#         single = client.get_post(test_posts[0].permalink)
#         print(f"   @{single.username} | {single.like_count} likes | {single.reply_count} replies")
#     except Exception as e:
#         print(f"   Error: {e}")

# print("\n" + "=" * 55)

## 節點 3：爬蟲執行

使用 `KeywordMonitor` 批次爬取多個關鍵字的貼文。

**執行流程：**
```
KeywordMonitor.run_round(keywords)
  → 逐一呼叫 client.search_posts(keyword)
    → _request() → GET /v1/threads/search?query=...
    → API JSON → ThreadsPost.from_api_response() 逐筆解析
  → seen_ids 去重（跨關鍵字 + 跨輪次）
  → 累積到 all_posts[keyword]
  → export_results() → CSV + JSON
```

In [6]:
# 5 地點 × 3 意圖關鍵字 + 7 機場/景點接送 = 22 credits / 輪
KEYWORDS = [
    # 日本地點：t1：直接包車意圖
    "日本包車", "沖繩包車", "富士山包車", "大阪包車", "北海道包車", "福岡包車",
    # 日本地點：t2：泛旅遊意圖
    "沖繩自由行", "沖繩旅遊", "日本自由行", "日本旅遊",
    "富士山自由行", "富士山旅遊",
    "大阪自由行", "大阪旅遊",
    "北海道自由行", "北海道旅遊",
    "福岡自由行", "福岡旅遊",
    
    # 韓國地點：t1：直接包車意圖
    "韓國包車", "首爾包車", "釜山包車", "濟州島包車",
    # 韓國地點：t2：泛旅遊意圖
    "韓國自由行", "韓國旅遊",
    "首爾自由行", "首爾旅遊",
    "釜山自由行", "釜山旅遊",
    "濟州島自由行", "濟州島旅遊",
    # t1：機場/景點接送
    # "沖繩機場接送", "那霸機場接送", "羽田機場接送",
    # "關西機場接送", "福岡機場接送", "新千歲機場接送",
    # "成田機場接送",
]
PROJECT_TAG = "japan_korea_charter"

In [7]:
# ── 爬蟲模式設定 ──
# MODE = "range"    → 自訂日期區間（手動回測、一次性分析）
# MODE = "schedule" → 模擬 Airflow 排程時段（測試 DAG 邏輯）
# MODE = "day"      → 單日爬取（快速測試）
MODE = "day"

# --- range 模式參數 ---
RANGE_START = "2026-04-02"
RANGE_END   = "2026-04-06"

# --- day 模式參數 ---
DAY_DATE = "2026-04-15"

# --- schedule 模式參數（模擬排程）---
SIMULATE_HOUR = 10  # 模擬時段: 10, 14, 18

# --- 動態輪數設定 ---
USE_ADAPTIVE = False   # True = run_adaptive (自動多輪), False = 固定輪數
FIXED_ROUNDS = 1       # 僅 USE_ADAPTIVE=False 時生效
MIN_ROUNDS = 2         # 僅 USE_ADAPTIVE=True 時生效
MAX_ROUNDS = 4
DUP_THRESHOLD = 0.80   # 當重複率超過此值，且新增數趨緩時，停止後續輪數（僅 adaptive 模式）

# ── get_scrape_params ──
from datetime import timedelta
from zoneinfo import ZoneInfo

TZ_TPE = ZoneInfo("Asia/Taipei")
SCHEDULE_HOURS = [10, 14, 18]

def get_scrape_params(execution_hour: int, ref_date=None):
    """
    根據排程時段計算 API 日期範圍 + post-filter 時間窗口。
    與 fetch_threads_posts.py 中的邏輯一致。
    """
    from datetime import date
    if ref_date is None:
        ref_date = date.today()

    today_str = ref_date.strftime('%Y-%m-%d')
    yesterday_str = (ref_date - timedelta(days=1)).strftime('%Y-%m-%d')
    ref_dt = datetime.combine(ref_date, datetime.min.time()).replace(tzinfo=TZ_TPE)

    closest_hour = min(SCHEDULE_HOURS, key=lambda h: abs(h - execution_hour))

    if closest_hour == 10:
        return {
            "start_date": yesterday_str,
            "end_date": today_str,
            "time_window": (ref_dt.replace(hour=0), ref_dt.replace(hour=10)),
            "label": f"10:00 slot | API {yesterday_str}~{today_str}, filter 00:00~10:00",
        }
    else:
        idx = SCHEDULE_HOURS.index(closest_hour)
        prev_hour = SCHEDULE_HOURS[idx - 1]
        return {
            "start_date": today_str,
            "end_date": today_str,
            "time_window": (ref_dt.replace(hour=prev_hour), ref_dt.replace(hour=closest_hour)),
            "label": f"{closest_hour}:00 slot | API {today_str}, filter {prev_hour}:00~{closest_hour}:00",
        }

# ── 根據 MODE 決定參數 ──
if MODE == "schedule":
    params = get_scrape_params(SIMULATE_HOUR)
    START_DATE = params["start_date"]
    END_DATE = params["end_date"]
    TIME_WINDOW = params["time_window"]
    print(f"[schedule] {params['label']}")
    print(f"  time_window: {TIME_WINDOW[0].isoformat()} ~ {TIME_WINDOW[1].isoformat()}")
elif MODE == "day":
    START_DATE = DAY_DATE
    END_DATE = DAY_DATE
    TIME_WINDOW = None
    print(f"[day] {DAY_DATE}")
else:  # range
    START_DATE = RANGE_START
    END_DATE = RANGE_END
    TIME_WINDOW = None
    print(f"[range] {RANGE_START} ~ {RANGE_END}")

print(f"  API params: start_date={START_DATE}, end_date={END_DATE}")
print(f"  adaptive: {'ON' if USE_ADAPTIVE else 'OFF'} (rounds={FIXED_ROUNDS if not USE_ADAPTIVE else f'{MIN_ROUNDS}~{MAX_ROUNDS}'})")
# ── 執行關鍵字爬蟲 ──
monitor = KeywordMonitor(client, output_dir=str(DATA_DIR))

print(f"爬取關鍵字: {len(KEYWORDS)} 組")
print(f"專案標記: {PROJECT_TAG}")
print(f"輸出目錄: {DATA_DIR}")
print(f"預計 API 請求數: {len(KEYWORDS)} 次/輪（每關鍵字 1 次）")
print(f"預計 credits 消耗: 約 {len(KEYWORDS)} credits/輪（以 Credits remaining 實際變化為準）\n")

# ── 執行爬蟲 ──
all_round_stats = []

if USE_ADAPTIVE:
    print(f"[adaptive] min={MIN_ROUNDS}, max={MAX_ROUNDS}, dup_threshold={DUP_THRESHOLD}")
    print("[adaptive] 用途：當新貼文趨緩且重複率升高時，自動停止後續輪數，降低 credits 消耗")
    all_posts, all_round_stats = monitor.run_adaptive(
        KEYWORDS,
        start_date=START_DATE,
        end_date=END_DATE,
        min_rounds=MIN_ROUNDS,
        max_rounds=MAX_ROUNDS,
        dup_threshold=DUP_THRESHOLD,
    )
else:
    for round_num in range(FIXED_ROUNDS):
        print(f"{'='*40} 第 {round_num+1}/{FIXED_ROUNDS} 輪 {'='*40}")
        _results, _stats = monitor.run_round(KEYWORDS, start_date=START_DATE, end_date=END_DATE)
        all_round_stats.append(_stats)
        print(f"  API回傳總筆數={_stats['api_total']}, 新增筆數={_stats['new_count']}, 重複率={_stats['dup_ratio']:.1%}")

# ── 合併所有關鍵字，加上 search_keyword 欄，去重 ──
rows = []
for kw, posts in monitor.all_posts.items():
    for p in posts:
        d = posts_to_dicts([p])[0]
        d['search_keyword'] = kw
        rows.append(d)

df_out = pd.DataFrame(rows).drop_duplicates(subset='post_id', keep='first')

# ── 時間窗口 post-filter（僅 schedule 模式）──
if TIME_WINDOW is not None:
    df_out['timestamp'] = pd.to_datetime(df_out['timestamp'], utc=True)
    start_utc = TIME_WINDOW[0].astimezone(ZoneInfo("UTC"))
    end_utc = TIME_WINDOW[1].astimezone(ZoneInfo("UTC"))
    before = len(df_out)
    df_out = df_out[(df_out['timestamp'] >= start_utc) & (df_out['timestamp'] < end_utc)].copy()
    print(f"\n[time_window filter] {before} -> {len(df_out)} posts")

print(f"\n── 爬取完成 ──")
print(f"  去重後貼文: {len(df_out)} 篇")
print(f"  總輪數: {len(all_round_stats)}")
total_api_results = sum(s.get('api_total', 0) for s in all_round_stats)
estimated_credits = len(KEYWORDS) * len(all_round_stats)
print(f"  API回傳總筆數(各輪加總): {total_api_results}")
print(f"  估算 credits 消耗: 約 {estimated_credits}（以 Credits remaining 差值為準）")

# ── 多輪策略覆蓋率提升評估  ── 
round_new = [s.get('new_count', 0) for s in all_round_stats]
if round_new:
    baseline = round_new[0]
    final_cum = sum(round_new)
    lift_pct = ((final_cum - baseline) / baseline * 100) if baseline > 0 else 0.0

    print(f"  第1輪新增: {baseline}")
    print(f"  多輪累積新增: {final_cum}")
    print(f"  覆蓋率提升(相對第1輪): {lift_pct:.1f}%")
    print("  各輪明細:")
    for i, s in enumerate(all_round_stats, 1):
        print(f"    R{i}: api={s['api_total']}, new={s['new_count']}, dup={s['dup_ratio']:.1%}")
#  ── 

# # ── 原始資料立即儲存 ──
# raw_csv = DATA_DIR / f"threads_{PROJECT_TAG}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
# raw_json = raw_csv.with_suffix('.json')
# df_out.to_csv(raw_csv, index=False, encoding='utf-8')
# save_json(df_out.to_dict(orient='records'), str(raw_json))
# print(f"  {raw_csv.name}")
# print(f"  {raw_json.name}")

2026-04-16 09:45:32 [INFO] ─── Search round: 2026-04-16 09:45:32 ───


[day] 2026-04-15
  API params: start_date=2026-04-15, end_date=2026-04-15
  adaptive: OFF (rounds=1)
爬取關鍵字: 30 組
專案標記: japan_korea_charter
輸出目錄: data
預計 API 請求數: 30 次/輪（每關鍵字 1 次）
預計 credits 消耗: 約 30 credits/輪（以 Credits remaining 實際變化為準）

======================================== 第 1/1 輪 ========================================


2026-04-16 09:45:35 [INFO] Credits remaining: 24113
2026-04-16 09:45:35 [INFO]   '日本包車': 9 results, 8 new (total unique: 8)
2026-04-16 09:45:38 [INFO] Credits remaining: 24112
2026-04-16 09:45:38 [INFO]   '沖繩包車': 9 results, 3 new (total unique: 3)
2026-04-16 09:45:44 [INFO] Credits remaining: 24111
2026-04-16 09:45:44 [INFO]   '富士山包車': 10 results, 8 new (total unique: 8)
2026-04-16 09:45:47 [INFO] Credits remaining: 24110
2026-04-16 09:45:47 [INFO]   '大阪包車': 9 results, 5 new (total unique: 5)
2026-04-16 09:45:52 [INFO] Credits remaining: 24109
2026-04-16 09:45:52 [INFO]   '北海道包車': 9 results, 6 new (total unique: 6)
2026-04-16 09:46:03 [INFO] Credits remaining: 24108
2026-04-16 09:46:03 [INFO]   '福岡包車': 9 results, 7 new (total unique: 7)
2026-04-16 09:46:08 [INFO] Credits remaining: 24107
2026-04-16 09:46:08 [INFO]   '沖繩自由行': 10 results, 9 new (total unique: 9)
2026-04-16 09:46:12 [INFO] Credits remaining: 24106
2026-04-16 09:46:12 [INFO]   '沖繩旅遊': 9 results, 6 new (total unique: 6)
202

  API回傳總筆數=252, 新增筆數=188, 重複率=25.4%

── 爬取完成 ──
  去重後貼文: 188 篇
  總輪數: 1
  API回傳總筆數(各輪加總): 252
  估算 credits 消耗: 約 30（以 Credits remaining 差值為準）
  第1輪新增: 188
  多輪累積新增: 188
  覆蓋率提升(相對第1輪): 0.0%
  各輪明細:
    R1: api=252, new=188, dup=25.4%


In [8]:
df_out

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,permalink,taken_at,timestamp,search_keyword
0,3875065127960267995_63081907205,weiling_0329,False,去沖繩包車不要找行腳沖繩\n原本找的是日本司機\n結果來的是大陸人\n這就算了還遲到30分鐘...,1187,137,8,1,88,False,https://www.threads.net/@weiling_0329/post/DXG...,1776163782,2026-04-14T10:49:42+00:00,日本包車
1,3876096286425038489_63429509529,miyokotour,True,我們家的日本包車\n現在正往「車齡六年內保證」的路上在轉型\n還需要一段時間🚗🚕🚌\n但也不遠了,0,0,0,0,0,False,https://www.threads.net/@miyokotour/post/DXKqQ...,1776286742,2026-04-15T20:59:02+00:00,日本包車
2,3876171494312711887_75229251425,tokyo_ezgo,False,早安，我的朋友們。\n\n早餐很重要，不管多忙碌，都一定要吃，大叔在這裡祝福大家都能擁有愉快...,130,179,21,0,0,False,https://www.threads.net/@tokyo_ezgo/post/DXK7W...,1776295671,2026-04-15T23:27:51+00:00,日本包車
3,3875749617091081566_43382360531,lovemore0925,False,網紅koL推薦 日本包車\n感謝三原、愛莉莎莎等人大力推薦\n隨著春天到來，日本櫻花季正式展...,0,1,0,0,0,False,https://www.threads.net/@lovemore0925/post/DXJ...,1776245380,2026-04-15T09:29:40+00:00,日本包車
4,3875719838825037761_41432450355,attomitravel.jp,False,🌸北海道櫻花季｜客製包車遊\n\n在北海道的櫻花季，景點之間往往距離遙遠，一台合法「綠牌」且...,5,2,0,0,0,False,https://www.threads.net/@attomitravel.jp/post/...,1776241830,2026-04-15T08:30:30+00:00,日本包車
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,3875795259162993135_65123382821,umi823,False,我從濟州島回來才發現\n去之前看到很多人分享必買的東西\n我存了一堆卻完全沒買！\n\n我竟...,11,2,0,0,0,False,https://www.threads.net/@umi823/post/DXJl0DcjaHv,1776250821,2026-04-15T11:00:21+00:00,濟州島旅遊
184,3875897501497323176_63085903605,chieh0803,False,這次濟州島讓我驚艷的食物之一\n光看這鍋真的不會想吃\n但是吃第一口才能理解為什麼韓國人都會...,31,10,0,0,52,False,https://www.threads.net/@chieh0803/post/DXJ9D4...,1776263009,2026-04-15T14:23:29+00:00,濟州島旅遊
185,3875892787284725615_63140331798,taoyun_0317,False,好想回去濟州島放空🥺,50,2,0,0,1,False,https://www.threads.net/@taoyun_0317/post/DXJ7...,1776262447,2026-04-15T14:14:07+00:00,濟州島旅遊
186,3875825830203618175_63381399056,chencheniam,False,人生就算被打倒，直接跑去濟州島。,2063,15,70,0,282,False,https://www.threads.net/@chencheniam/post/DXJs...,1776254465,2026-04-15T12:01:05+00:00,濟州島旅遊


---

## 節點 5：載入資料 & 資料檢視

讀取爬取結果 CSV，執行基礎清洗與資料概覽。

> **語言策略備註：** 本次爬取未使用程式語言過濾。
> 改以關鍵字設計避開日文——使用繁體中文特有字（如「裝備」的「裝」 vs 日文「装」），
> 自然排除多數日文內容。若未來資料出現大量非繁中貼文，可重新啟用語言過濾。

In [9]:
# ── 轉換層 ──
# df_out = pd.read_csv(DATA_DIR / raw_csv.name, encoding='utf-8')

df = df_out.copy()

df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# ── taken_at 轉台灣時間（新增欄位）──
df['taken_at_tpe'] = (
    pd.to_datetime(df['taken_at'], unit='s', utc=True)
      .dt.tz_convert('Asia/Taipei')
      .dt.strftime('%Y-%m-%d %H:%M:%S')
)

# post_date 改以台灣日期為準（讓日期過濾以台灣時區為基準，避免 UTC 跨日誤差）
df['post_date'] = pd.to_datetime(df['taken_at_tpe']).dt.date

df['text_length'] = df['text'].fillna('').str.len()
engagement_cols = ['like_count', 'reply_count', 'repost_count', 'quote_count', 'reshare_count']
df['total_engagement'] = df[engagement_cols].sum(axis=1)

# 日期範圍過濾（post_date 為台灣日期，與 START_DATE/END_DATE 對齊）
from datetime import date
filter_start = date.fromisoformat(START_DATE)
filter_end = date.fromisoformat(END_DATE)
x_df = df[(df['post_date'] < filter_start) | (df['post_date'] > filter_end)].copy()
df = df[(df['post_date'] >= filter_start) & (df['post_date'] <= filter_end)].copy()
print(f"[{MODE} filter] 篩選台灣日期: {START_DATE}" + (f" ~ {END_DATE}" if START_DATE != END_DATE else ""))

# 標記每篇貼文命中了哪些關鍵字
for kw in KEYWORDS:
    df[f'has_{kw}'] = df['text'].fillna('').str.contains(kw, case=False)
df['matched_keywords'] = df[[f'has_{kw}' for kw in KEYWORDS]].sum(axis=1)

# 意圖分類：文中有包車/租車/司機/接送 → t1，其餘 → t2
df['audience_type'] = np.where(
    df['text'].fillna('').str.contains(r'包車|租車|司機|接送', regex=True),
    't1', 't2'
)

print(f"日期篩選後剩餘總篇數: {len(df)}")
print(f"抓到不在設定日期範圍內的貼文: {len(x_df)} 篇")

print(f"  t1 (包車意圖): {(df['audience_type']=='t1').sum()} 篇")
print(f"  t2 (旅遊意圖): {(df['audience_type']=='t2').sum()} 篇")

# ── 地點驗證：篩出不含任何目標地點的貼文 ──
LOCATIONS = ["日本", "沖繩", "富士山", "大阪", "北海道", "福岡", "韓國", "首爾", "釜山", "濟州島"]
location_pattern = '|'.join(LOCATIONS)
has_location = df['text'].fillna('').str.contains(location_pattern, case=False)
only_car_service = df[~has_location].copy()
df = df[has_location].copy()

print(f"── 地點驗證 ──")
print(f"  含目標地點: {len(df)} 篇")
print(f"  不含目標地點 (only_car_service): {len(only_car_service)} 篇")


# ── 商業文判定（COMMERCIAL_TERMS）──
import re
COMMERCIAL_TERMS = [
    "LINE ID", "歡迎諮詢", "招募", "互追", "漲粉",
    "立即預訂", "車隊", "限時", "超時費", "客製",
    "經營", "急單", "服務品質", "客人", "提前預約",
    "免費幫忙", "免費協助", "為您", "領取", "公司",
    "留言", "傳給你",
]
commercial_pattern = '|'.join(COMMERCIAL_TERMS)

def is_commercial(text: str) -> bool:
    return bool(re.search(commercial_pattern, text or '', flags=re.IGNORECASE))


[day filter] 篩選台灣日期: 2026-04-15
日期篩選後剩餘總篇數: 144
抓到不在設定日期範圍內的貼文: 44 篇
  t1 (包車意圖): 35 篇
  t2 (旅遊意圖): 109 篇
── 地點驗證 ──
  含目標地點: 128 篇
  不含目標地點 (only_car_service): 16 篇


In [10]:
x_df

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,permalink,taken_at,timestamp,search_keyword,taken_at_tpe,post_date,text_length,total_engagement
0,3875065127960267995_63081907205,weiling_0329,False,去沖繩包車不要找行腳沖繩\n原本找的是日本司機\n結果來的是大陸人\n這就算了還遲到30分鐘...,1187,137,8,1,88,False,https://www.threads.net/@weiling_0329/post/DXG...,1776163782,2026-04-14 10:49:42+00:00,日本包車,2026-04-14 18:49:42,2026-04-14,136,1421
1,3876096286425038489_63429509529,miyokotour,True,我們家的日本包車\n現在正往「車齡六年內保證」的路上在轉型\n還需要一段時間🚗🚕🚌\n但也不遠了,0,0,0,0,0,False,https://www.threads.net/@miyokotour/post/DXKqQ...,1776286742,2026-04-15 20:59:02+00:00,日本包車,2026-04-16 04:59:02,2026-04-16,45,0
2,3876171494312711887_75229251425,tokyo_ezgo,False,早安，我的朋友們。\n\n早餐很重要，不管多忙碌，都一定要吃，大叔在這裡祝福大家都能擁有愉快...,130,179,21,0,0,False,https://www.threads.net/@tokyo_ezgo/post/DXK7W...,1776295671,2026-04-15 23:27:51+00:00,日本包車,2026-04-16 07:27:51,2026-04-16,104,330
12,3777738195581710960_78811101885,japancharter.link,False,🇯🇵 日本接送機 / 日本包車/一價全含\n成田、羽田、關西、中部、札幌新千歲、沖繩那霸\n...,10,39,2,0,30,False,https://www.threads.net/@japancharter.link/pos...,1764561508,2025-12-01 03:58:28+00:00,富士山包車,2025-12-01 11:58:28,2025-12-01,501,81
13,3875972584346063780_64232195628,shen_ricky,False,大家好～～\n想問一下有沒有斗六或雲林的朋友\n4/22 約23:40左右的飛機抵達桃園機場...,8,20,0,0,1,False,https://www.threads.net/@shen_ricky/post/DXKOI...,1776271959,2026-04-15 16:52:39+00:00,富士山包車,2026-04-16 00:52:39,2026-04-16,118,29
14,3876067278719225958_65624396836,monkey9751,False,洗腎洗了7年\n雖然目前身體大致上還沒什麼問題\n不過總覺得不帶家人出去走走的話以後不一定有...,15,2,0,0,0,False,https://www.threads.net/@monkey9751/post/DXKjq...,1776283248,2026-04-15 20:00:48+00:00,富士山包車,2026-04-16 04:00:48,2026-04-16,124,17
15,3873812174540329300_65604519507,miniya1116,False,想問日本包車4天三夜5人價格\n\n想去富士山，淺草，秋葉原一些景點,2,10,0,0,0,False,https://www.threads.net/@miniya1116/post/DXCi6...,1776014418,2026-04-12 17:20:18+00:00,富士山包車,2026-04-13 01:20:18,2026-04-13,32,12
16,3694024663598320998_74715623370,maazitravel_official,True,富士山包車一日遊！邁齊都幫你安排好了，\n直接躺著玩、躺著拍！\n只要一天\n全富士山最美打...,156,557,4,5,339,False,https://www.threads.net/@maazitravel_official/...,1754582077,2025-08-07 15:54:37+00:00,富士山包車,2025-08-07 23:54:37,2025-08-07,162,1061
20,3875110927041656964_64626156136,kamelio.tw,False,0天京都＋大阪…帶著一歲的寶寶 👶\n六月會跟家人去日本玩幾天。\n\n目前行程大致確定了（...,9,36,0,0,11,False,https://www.threads.net/@kamelio.tw/post/DXHKN...,1776169242,2026-04-14 12:20:42+00:00,大阪包車,2026-04-14 20:20:42,2026-04-14,293,56
21,3875090782714262637_75305418244,innn_tw,False,DIVE們都在哪裡～～～👀💖\nIVE 要去日本開演唱會啦🇯🇵✨\n\n最近小編真的被私訊愛...,21,14,1,0,1,False,https://www.threads.net/@innn_tw/post/DXHFokoEvRt,1776166840,2026-04-14 11:40:40+00:00,大阪包車,2026-04-14 19:40:40,2026-04-14,456,37


In [11]:
# 排除商家文
df['is_commercial'] = df['text'].fillna('').apply(is_commercial)
post_is_commercial = df[df['is_commercial']].copy()
df_clean = df[~df['is_commercial']].copy()

print(f"商家文過濾: {len(df)} → {len(df_clean)} 篇（排除 {len(post_is_commercial)} 篇商家文）")

商家文過濾: 128 → 106 篇（排除 22 篇商家文）


In [12]:
post_is_commercial

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,has_韓國旅遊,has_首爾自由行,has_首爾旅遊,has_釜山自由行,has_釜山旅遊,has_濟州島自由行,has_濟州島旅遊,matched_keywords,audience_type,is_commercial
3,3875749617091081566_43382360531,lovemore0925,False,網紅koL推薦 日本包車\n感謝三原、愛莉莎莎等人大力推薦\n隨著春天到來，日本櫻花季正式展...,0,1,0,0,0,False,...,False,False,False,False,False,False,False,2,t1,True
4,3875719838825037761_41432450355,attomitravel.jp,False,🌸北海道櫻花季｜客製包車遊\n\n在北海道的櫻花季，景點之間往往距離遙遠，一台合法「綠牌」且...,5,2,0,0,0,False,...,False,False,False,False,False,False,False,1,t1,True
5,3875564774902784779_75229251425,tokyo_ezgo,False,最近詐騙太多了，所以再來做一次自我介紹。\n\n我是Tokyo EZGO大叔，在日本成田經營...,166,68,45,0,16,False,...,False,False,False,False,False,False,False,0,t1,True
23,3875753912528122363_43382360531,lovemore0925,False,網紅koL推薦 日本包車\n感謝三原、愛莉莎莎等人大力推薦\n隨著春天到來，日本櫻花季正式展...,0,0,0,0,0,False,...,False,False,False,False,False,False,False,2,t1,True
34,3875751641983553822_43382360531,lovemore0925,False,網紅koL推薦 日本包車\n感謝三原、愛莉莎莎等人大力推薦\n隨著春天到來，日本櫻花季正式展...,0,0,0,0,0,False,...,False,False,False,False,False,False,False,2,t1,True
35,3875671011899814057_77462381556,kyushu_tule_taxi,False,4.15 九州自然動物園\n八點四十到 買到十點四十 很OK\n包車就是好\n\n上下車地點...,1,0,0,0,0,False,...,False,False,False,False,False,False,False,0,t1,True
36,3875750387467276412_43382360531,lovemore0925,False,九州三原推薦包車網紅koL推薦 日本包車\n感謝三原、愛莉莎莎等人大力推薦\n隨著春天到來，...,0,0,0,0,0,False,...,False,False,False,False,False,False,False,2,t1,True
43,3875565235722094253_33973516366,lucky8085888,False,🇯🇵沖繩新開🔥東京人氣日料｜徵台灣、香港創作者\n最近要來沖繩的先看這篇👇\n直接請你吃😆\...,33,20,0,0,12,False,...,False,False,False,False,False,False,False,2,t2,True
68,3875863057470195792_67720916528,jhg_japaholicgirls,False,徵求 經常去日本玩的人🇯🇵\n\n【合作內容】Threads文一篇\n\n【貼文主題】\n日...,70,58,0,0,25,False,...,False,False,False,False,False,False,False,1,t2,True
73,3875825651945827215_78569616270,steven.wang1224,False,✈️ 第一次飛日本，到底該選「東京」還是「大阪」？\n\n短短 12 秒，幫你解決人生最難的...,0,0,0,0,0,False,...,False,False,False,False,False,False,False,2,t2,True


In [13]:
only_car_service

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,has_韓國自由行,has_韓國旅遊,has_首爾自由行,has_首爾旅遊,has_釜山自由行,has_釜山旅遊,has_濟州島自由行,has_濟州島旅遊,matched_keywords,audience_type
10,3875849446769114128_67637351443,abc_abc2221,False,想請問包車多少錢？,0,1,0,0,0,True,...,False,False,False,False,False,False,False,False,0,t1
11,3875919718918337588_63075019430,_tzu.7,False,"想問有沒有5/16,新竹包車當天來回的🥹🥹",10,14,0,0,0,False,...,False,False,False,False,False,False,False,False,0,t1
18,3875899706434646164_66807103883,iswian_jeou,False,我蹲一個5/16台北🔁高雄當天來回的包車😣,8,14,0,0,2,False,...,False,False,False,False,False,False,False,False,0,t1
29,3875588957102166563_77478221748,fengjian25109,False,小樽，登別，旭川，函館，道東包車可以私訊。價格合理。,5,0,0,0,0,False,...,False,False,False,False,False,False,False,False,0,t1
31,3875928898052190589_69712336455,cooogdk,False,你好想了解包車,1,1,0,0,0,True,...,False,False,False,False,False,False,False,False,0,t1
38,3875596340177418668_64357263812,boy90170,False,工欲善其事 必先利其器\n這工具太強大了,714,69,41,5,1430,False,...,False,False,False,False,False,False,False,False,0,t2
59,3875883177655983928_78986410515,parasan73,False,安達結希くんの事件\n\n父親が逮捕されたと。\nネットでは24歳と言われていたけどバツイチ...,610,128,14,1,46,False,...,False,False,False,False,False,False,False,False,0,t2
60,3875934994707996280_63429460481,zkeplqn195,False,求各位大神幫我旅遊健檢！\n第一次帶長輩出國怕被碎碎念！,6,13,0,0,1,False,...,False,False,False,False,False,False,False,False,0,t2
67,3875920461385770822_77818235560,homo9844,False,我真的要氣死！！\n去一趟名古屋回來發現我根本巨型芥菜！！\n當初在那邊自以為很會比價，訂完...,27,15,14,0,94,False,...,False,False,False,False,False,False,False,False,0,t2
82,3875797335470084622_46410471043,__7.05,False,預計12/19-12/26旅遊 人數4\n理想地點在地鐵站附近，靠近心齋橋、難波\n希望是兩...,2,2,0,0,0,False,...,False,False,False,False,False,False,False,False,0,t2


In [14]:
df_clean

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,has_韓國旅遊,has_首爾自由行,has_首爾旅遊,has_釜山自由行,has_釜山旅遊,has_濟州島自由行,has_濟州島旅遊,matched_keywords,audience_type,is_commercial
6,3875464153851444233_80489010954,tipsytrip_,False,日本旅遊🧳包車接送機服務🚗,0,1,0,0,0,True,...,False,False,False,False,False,False,False,1,t1,False
7,3875859900409263345_63508128396,hanlo.hsueh,False,覺得自己太偉大了🥹\n含六位長輩的十人日本自助團順利結束，廣島進關西回，前兩晚住廣島後四晚住...,12,1,0,0,0,False,...,False,False,False,False,False,False,False,0,t1,False
8,3875859078267056507_63306482951,sunnypuddingchen,False,9人座包車40000日幣（8小時/日文司機）\n我跟行腳沖繩預訂的，但台灣客服很ㄎㄧㄤ，要個...,1,1,0,0,1,True,...,False,False,False,False,False,False,False,0,t1,False
9,3875824625624866446_63306482951,sunnypuddingchen,False,繼前幾篇，通知大家逛國際通一定要走到唐吉軻德，原因大公開⋯(沖繩鬧事 之 不知道第幾篇續集）...,22,8,2,0,22,False,...,False,False,False,False,False,False,False,0,t1,False
17,3875901367211171184_70846666826,car4tourjp,False,#仙台包車推薦\n\n今次帶的包車團來到 #仙台，#白石川堤一目千本櫻，影片是今天（4月15...,2,1,0,1,3,False,...,False,False,False,False,False,False,False,0,t1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,3875795259162993135_65123382821,umi823,False,我從濟州島回來才發現\n去之前看到很多人分享必買的東西\n我存了一堆卻完全沒買！\n\n我竟...,11,2,0,0,0,False,...,False,False,False,False,False,False,False,0,t2,False
184,3875897501497323176_63085903605,chieh0803,False,這次濟州島讓我驚艷的食物之一\n光看這鍋真的不會想吃\n但是吃第一口才能理解為什麼韓國人都會...,31,10,0,0,52,False,...,False,False,False,False,False,False,False,0,t2,False
185,3875892787284725615_63140331798,taoyun_0317,False,好想回去濟州島放空🥺,50,2,0,0,1,False,...,False,False,False,False,False,False,False,0,t2,False
186,3875825830203618175_63381399056,chencheniam,False,人生就算被打倒，直接跑去濟州島。,2063,15,70,0,282,False,...,False,False,False,False,False,False,False,0,t2,False


In [15]:
# ── 資料觀察 ──
print("=== 資料概覽 ===")
print(f"  貼文數: {len(df_clean)}")
print(f"  不重複帳號: {df_clean['username'].nunique()}")
print(f"  時間範圍: {df_clean['post_date'].min():%Y-%m-%d} ~ {df_clean['post_date'].max():%Y-%m-%d}")
print(f"  is_reply 分布: {df_clean['is_reply'].value_counts().to_dict()}")

print(f"\n=== search_keyword 來源分布 ===")
print(df_clean['search_keyword'].value_counts().to_string())

print(f"\n=== 互動指標描述統計 ===")
print(df_clean[engagement_cols + ['total_engagement']].describe().round(1).to_string())

=== 資料概覽 ===
  貼文數: 106
  不重複帳號: 100
  時間範圍: 2026-04-15 ~ 2026-04-15
  is_reply 分布: {False: 101, True: 5}

=== search_keyword 來源分布 ===
search_keyword
濟州島旅遊     8
韓國旅遊      8
大阪自由行     7
釜山自由行     7
日本自由行     7
釜山旅遊      6
福岡旅遊      5
北海道旅遊     5
大阪旅遊      5
福岡自由行     5
沖繩自由行     5
日本旅遊      4
首爾包車      4
韓國自由行     4
首爾自由行     4
北海道自由行    3
沖繩旅遊      3
北海道包車     3
沖繩包車      2
釜山包車      2
首爾旅遊      2
大阪包車      2
日本包車      2
韓國包車      1
富士山包車     1
濟州島自由行    1

=== 互動指標描述統計 ===
       like_count  reply_count  repost_count  quote_count  reshare_count  total_engagement
count       106.0        106.0         106.0        106.0          106.0             106.0
mean         81.4         10.6           5.6          0.2           77.7             175.5
std         247.7         28.1          22.4          1.2          263.9             492.8
min           0.0          0.0           0.0          0.0            0.0               0.0
25%           4.0          0.0           0.0          0.0        

In [16]:
# # ── 輸出清洗後 CSV/JSON ──
# csv_path = DATA_DIR / f"threads_{PROJECT_TAG}_clean_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
# json_path = csv_path.with_suffix('.json')

# df_clean.to_csv(csv_path, index=False, encoding='utf-8')
# save_json(df_clean.to_dict(orient='records'), str(json_path))

# print(f"✔ {csv_path.name}")
# print(f"✔ {json_path.name}")

In [ ]:
# ── 上傳到 Google Sheets ──
import gspread
from google.oauth2.service_account import Credentials

# 篩選欄位（含 taken_at_tpe 台灣時間）
ouput_df = df_clean[['username', 'post_date', 'text', 'permalink', 'search_keyword', 'audience_type']].copy()
ouput_df['scrape_date'] = datetime.now().strftime('%Y-%m-%d')
ouput_df['taken_at_tpe'] = df_clean['taken_at_tpe'].values

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
CREDS_PATH = Path("ads-de-01-757fc521ef01.json")
SHEET_ID = "1ld3Mt6ZbT1uNSidxYZN94EuIaHL8AIIw2gNZTiFH4Jw"

creds = Credentials.from_service_account_file(CREDS_PATH, scopes=SCOPES)
gc = gspread.authorize(creds)

sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1

# ── 安全寫入：明確定位最後一行，不依賴 append_rows 的表格偵測 ──
existing = ws.get_all_values()
next_row = len(existing) + 1

if not existing or not any(str(cell).strip() for cell in existing[0]):
    ws.update(range_name='A1', values=[ouput_df.columns.tolist()], value_input_option="RAW")
    next_row = 2
    print(f"[init] 已寫入標題列")

data_rows = ouput_df.astype(str).values.tolist()
ws.update(range_name=f'A{next_row}', values=data_rows, value_input_option="USER_ENTERED")

print(f"[OK] 已寫入 {len(data_rows)} 筆到 A{next_row}~A{next_row + len(data_rows) - 1}")
print(f"  Sheet 目前共 {next_row + len(data_rows) - 1} 行（含標題）")
print(f"  https://docs.google.com/spreadsheets/d/{SHEET_ID}")

In [18]:
df_clean

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,has_韓國旅遊,has_首爾自由行,has_首爾旅遊,has_釜山自由行,has_釜山旅遊,has_濟州島自由行,has_濟州島旅遊,matched_keywords,audience_type,is_commercial
6,3875464153851444233_80489010954,tipsytrip_,False,日本旅遊🧳包車接送機服務🚗,0,1,0,0,0,True,...,False,False,False,False,False,False,False,1,t1,False
7,3875859900409263345_63508128396,hanlo.hsueh,False,覺得自己太偉大了🥹\n含六位長輩的十人日本自助團順利結束，廣島進關西回，前兩晚住廣島後四晚住...,12,1,0,0,0,False,...,False,False,False,False,False,False,False,0,t1,False
8,3875859078267056507_63306482951,sunnypuddingchen,False,9人座包車40000日幣（8小時/日文司機）\n我跟行腳沖繩預訂的，但台灣客服很ㄎㄧㄤ，要個...,1,1,0,0,1,True,...,False,False,False,False,False,False,False,0,t1,False
9,3875824625624866446_63306482951,sunnypuddingchen,False,繼前幾篇，通知大家逛國際通一定要走到唐吉軻德，原因大公開⋯(沖繩鬧事 之 不知道第幾篇續集）...,22,8,2,0,22,False,...,False,False,False,False,False,False,False,0,t1,False
17,3875901367211171184_70846666826,car4tourjp,False,#仙台包車推薦\n\n今次帶的包車團來到 #仙台，#白石川堤一目千本櫻，影片是今天（4月15...,2,1,0,1,3,False,...,False,False,False,False,False,False,False,0,t1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,3875795259162993135_65123382821,umi823,False,我從濟州島回來才發現\n去之前看到很多人分享必買的東西\n我存了一堆卻完全沒買！\n\n我竟...,11,2,0,0,0,False,...,False,False,False,False,False,False,False,0,t2,False
184,3875897501497323176_63085903605,chieh0803,False,這次濟州島讓我驚艷的食物之一\n光看這鍋真的不會想吃\n但是吃第一口才能理解為什麼韓國人都會...,31,10,0,0,52,False,...,False,False,False,False,False,False,False,0,t2,False
185,3875892787284725615_63140331798,taoyun_0317,False,好想回去濟州島放空🥺,50,2,0,0,1,False,...,False,False,False,False,False,False,False,0,t2,False
186,3875825830203618175_63381399056,chencheniam,False,人生就算被打倒，直接跑去濟州島。,2063,15,70,0,282,False,...,False,False,False,False,False,False,False,0,t2,False
